In [15]:
import polars as pl

TARGET_PATH = "/Users/intuitivecode/Code/deltastreams/data/tables/bronze_liquidity"
df = pl.read_delta(TARGET_PATH)

print(df.sort("_ingested_at", descending=True).head(5))

import polars as pl

df_analyzed = df.with_columns([
    pl.col("value")
    .str.json_decode(dtype=pl.Struct([pl.Field("message", pl.String)]))
    .struct.field("message")
    .alias("inner_message")
])

df_counts = df_analyzed.with_columns([
    pl.when(pl.col("inner_message").str.contains('"type":"ticker"')).then(pl.lit("Standard Ticker"))
    .when(pl.col("inner_message").str.contains('"type":"subscriptions"')).then(pl.lit("System Subscription"))
    .otherwise(pl.lit("Other System Packet"))
    .alias("Message_Type")
])

print(" --- BRONZE PRE-ATTACK METRICS ---")
print(f"Total Combined Rows in Delta Lake: {df_counts.height}")
print(f"Total Partitions Active: {df_counts.select('partition').n_unique()}")
print(f"Current Maximum Offset Captured: {df_counts.select('offset').max().item()}\n")

print(" --- MESSAGE VOLUME SPLIT ---")
print(df_counts.group_by("Message_Type").agg(pl.len().alias("Count")).sort("Count", descending=True))


shape: (5, 5)
┌───────────┬────────────────────────────────┬───────────┬────────┬────────────────────────────────┐
│ kafka_key ┆ value                          ┆ partition ┆ offset ┆ _ingested_at                   │
│ ---       ┆ ---                            ┆ ---       ┆ ---    ┆ ---                            │
│ str       ┆ str                            ┆ i32       ┆ i64    ┆ str                            │
╞═══════════╪════════════════════════════════╪═══════════╪════════╪════════════════════════════════╡
│ null      ┆ {"file":"data/market.log","hos ┆ 0         ┆ 5683   ┆ 2026-11-05T22:51:40.340559+00: │
│           ┆ …                              ┆           ┆        ┆ …                              │
│ null      ┆ {"file":"data/market.log","hos ┆ 0         ┆ 5682   ┆ 2026-11-05T22:51:40.073166+00: │
│           ┆ …                              ┆           ┆        ┆ …                              │
│ null      ┆ {"file":"data/market.log","hos ┆ 0         ┆ 5681   ┆ 2026-11-0